# Lab 21.3: A Mini Hermeneutic Circle Test

Chapter 21 data case study. Explain 100 test applicants of a German Credit random forest with KernelSHAP, LIME and HermAI 0.1.8 (two generators), add two controls (random attributions and sign-flipped KernelSHAP), and compute the Hermeneutic Circle Test profile: Horizon Plausibility (HP), Claim Verification Rate (CVR), Part-Whole Agreement (PWA) and the Hermeneutic Horizon Radius (HHR), and Hermeneutic Gain (HG) with three machine readers.

`hct_mini.py` is a simplified teaching re-implementation of the test defined in Seker (2026), *Machine Learning and Knowledge Extraction* 8(11), 381. It is not the reference implementation, which is released with the paper. *Disclosure: the author of the book designed the test and maintains HermAI.*

Total run time: about 8 minutes on a laptop CPU (explanations about 3 minutes, profile about 4 minutes).

In [1]:
import numpy as np, pandas as pd
import hct_mini as h
from sklearn.metrics import roc_auc_score
pd.set_option("display.width", 200)
S = h.setup(0)
f, Xte, cols, cb = S["f"], S["Xte"].values, S["cols"], S["codebook"]
yhat = (f(Xte) >= 0.5).astype(int)
print("train / fit / calibration / test:", len(S["Xtr"]), len(S["Xfit"]), len(S["Xcal"]), len(Xte))
print("test AUC:", round(roc_auc_score(S["yte"], f(Xte)), 3), "| predicted bad risk:", yhat.sum(), "of", len(Xte))
print("largest absolute correlation between two features:", round(h.max_abs_corr(S["Xfit"]), 2))

train / fit / calibration / test: 700 525 175 300
test AUC: 0.787 | predicted bad risk: 45 of 300
largest absolute correlation between two features: 0.59


**Checkpoint 1:** 700 / 525 / 175 / 300 rows; test AUC 0.787; 45 of 300 test applicants predicted bad risk (probability at least 0.5); largest correlation 0.59, below the 0.9 the paper uses to group features into meaning units, so every unit here is a single feature.

## 1. Explain 100 applicants (about 3 minutes)

In [2]:
idx = h.pick_cases(S, 100)
print("explained:", len(idx), "| of which predicted bad:", yhat[idx].sum())
out = h.explain_all(S, idx)

explained: 100 | of which predicted bad: 45


  explained 100 cases with KernelSHAP


  explained 100 cases with LIME


  explained 100 cases with HermAI 0.1.8


  explained 100 cases with HermAI 0.1.8 (cov)


**Checkpoint 2:** 100 applicants, 45 predicted bad (all of them) and 55 predicted good.

## 2. The profile (about 4 minutes)

In [3]:
prof, per, H, W, real = h.hct_profile(S, idx, out)
print("real test records inside the horizon:", round(real, 2))
prof[["HP", "HPdist", "HPtype", "CVR", "CVR_vector", "CVR_marginal", "PWA", "GI", "AR", "HG", "HGmin", "PGI_marginal", "HCT"]].round(3)

  profile done: KernelSHAP


  profile done: LIME


  profile done: HermAI 0.1.8


  profile done: HermAI 0.1.8 (cov)


  profile done: Random


  profile done: Sign-flip
real test records inside the horizon: 0.92


,HP,HPdist,HPtype,CVR,CVR_vector,CVR_marginal,PWA,GI,AR,HG,HGmin,PGI_marginal,HCT
explainer,,,,,,,,,,,,,
KernelSHAP,0.955,0.961,0.990,0.927,0.927,0.930,0.457,0.48,0.26,0.123,0.049,0.187,0.376
LIME,0.001,0.721,0.001,0.773,0.773,0.783,0.463,0.55,0.25,0.104,-0.004,0.176,0.000
HermAI 0.1.8,0.000,0.909,0.000,0.287,0.790,0.297,0.410,0.43,0.37,0.102,0.063,0.150,0.000
HermAI 0.1.8 (cov),0.000,0.894,0.000,0.297,0.823,0.300,0.450,0.51,0.33,0.062,0.005,0.164,0.000
Random,NaN,NaN,NaN,0.227,0.227,0.230,0.040,0.00,0.98,-0.057,-0.159,0.056,NaN
Sign-flip,0.955,0.961,0.990,0.003,0.003,0.007,0.003,0.00,1.00,0.123,0.049,0.187,0.027


**Checkpoint 3 (compare with the chapter):** HP 0.96 for KernelSHAP and 0.00 for LIME and both HermAI generators, against 0.92 for real records; CVR 0.93 (KernelSHAP), 0.77 (LIME), 0.29 and 0.30 (HermAI sentences), 0.23 (random), 0.00 (sign-flip); PWA between 0.41 and 0.46 for the four explainers, 0.04 for random and 0.01 for sign-flip; HG 0.12 for KernelSHAP and its sign-flipped copy alike; PGI 0.187 for both KernelSHAP and the sign-flip control.

Read three things. First, `CVR_vector` for HermAI (0.79 and 0.82) is far above `CVR` (0.29 and 0.30): the attribution *vector*, read as contributions, mostly passes; the *sentences* the library writes mostly fail. Second, PGI cannot tell KernelSHAP from its sign-flipped copy; CVR and PWA can. Third, HP falls to zero for every explainer that samples continuous noise, because every numeric feature in German Credit is an integer.

In [4]:
prof[["slope_CVR", "inversion", "CF_found", "CF_valid", "CF_changed", "PWA_half", "PWA_quarter", "PWA_eighth"]].round(3)

,slope_CVR,inversion,CF_found,CF_valid,CF_changed,PWA_half,PWA_quarter,PWA_eighth
explainer,,,,,,,,
KernelSHAP,NaN,NaN,NaN,NaN,NaN,0.357,0.367,0.370
LIME,NaN,NaN,NaN,NaN,NaN,0.347,0.350,0.340
HermAI 0.1.8,0.603,0.667,1.0,0.0,11.56,0.330,0.307,0.323
HermAI 0.1.8 (cov),0.670,0.690,1.0,0.0,11.47,0.337,0.313,0.327
Random,NaN,NaN,NaN,NaN,NaN,0.033,0.030,0.037
Sign-flip,NaN,NaN,NaN,NaN,NaN,0.000,0.017,0.020


**Checkpoint 4:** HermAI sentences pass the literal slope reading 0.60 and 0.67 of the time; 0.67 and 0.69 of them state a slope whose sign is opposite to the feature's contribution; counterfactuals are always found, change about 11.5 features and are never inside the horizon. PWA falls from the global scale to the narrower wholes for every explainer.

## 3. The featured applicant

In [5]:
k = list(idx).index(77)
x = Xte[77]
print("predicted risk:", round(f(x[None])[0], 3))
print({c: (cb[c][int(v)] if c in cb else v) for c, v in zip(cols, x)})
for name in ["KernelSHAP", "LIME", "HermAI 0.1.8"]:
    c = per[name][k]
    print(f"\n{name}: PWA by scale {np.round(c['pwa'], 2)}, radius {c['hhr']}")
    for (j, s), (ok, d) in zip(c["claims"], c["results"]):
        print(f"  says {cols[j]} {'raised' if s > 0 else 'lowered'} the risk -> on-horizon effect {d:+.3f}: {'verified' if ok else 'fails'}")
for s in h.SCALES:
    g, yw = W.account(x, s)
    print(f"whole at scale {s}: top 3 {[ (cols[j], round(g[j], 3)) for j in h.top3(g)]}")

predicted risk: 0.668
{'checking_status': '<0', 'duration': np.float64(15.0), 'credit_history': 'no credits/all paid', 'purpose': 'new car', 'credit_amount': np.float64(950.0), 'savings_status': '<100', 'employment': '>=7', 'installment_commitment': np.float64(4.0), 'personal_status': 'male single', 'other_parties': 'none', 'residence_since': np.float64(3.0), 'property_magnitude': 'car', 'age': np.float64(33.0), 'other_payment_plans': 'none', 'housing': 'rent', 'existing_credits': np.float64(2.0), 'job': 'skilled', 'num_dependents': np.float64(2.0), 'own_telephone': 'none', 'foreign_worker': 'yes'}

KernelSHAP: PWA by scale [0.33 0.33 1.   0.67], radius 0.25
  says checking_status raised the risk -> on-horizon effect +0.130: verified
  says credit_history raised the risk -> on-horizon effect +0.073: verified
  says credit_amount raised the risk -> on-horizon effect +0.064: verified

LIME: PWA by scale [0.67 0.67 0.67 0.33], radius 1.0
  says checking_status raised the risk -> on-horizo

**Checkpoint 5:** predicted risk 0.668. KernelSHAP's three sentences (checking account, credit history, loan amount raised the risk) are all verified; they agree with the global whole on one of three features (PWA 0.33) but with the whole fitted to the nearest quarter of applicants on all three (PWA 1.0), so the radius is 0.25. HermAI's three sentences all fail.

## 4. Read the data behind a strange part

In [6]:
tr = S["Xtr"].copy(); tr["bad"] = S["ytr"]
tr["credit_history"] = tr["credit_history"].map(lambda v: cb["credit_history"][int(v)])
print(tr.groupby("credit_history")["bad"].agg(["mean", "count"]).round(3))
print(tr.groupby(pd.cut(tr["credit_amount"], [0, 1000, 2000, 4000, 8000, 20000]))["bad"].agg(["mean", "count"]).round(3))

                                 mean  count
credit_history                              
all paid                        0.641     39
critical/other existing credit  0.152    198
delayed previously              0.354     65
existing paid                   0.313    374
no credits/all paid             0.625     24
                mean  count
credit_amount              
(0, 1000]      0.387     75
(1000, 2000]   0.262    225
(2000, 4000]   0.233    232
(4000, 8000]   0.339    124
(8000, 20000]  0.591     44


**Checkpoint 6:** in the training data, 0.625 of the 24 applicants coded "no credits/all paid" were bad risks, against about 0.30 overall; loans up to 1,000 DM had a bad rate of 0.387. The model learned both patterns. Whether the first is a fact about borrowers or about how this old dataset was coded is a question about the whole, not the part.

## Your turn

1. Set `h.ALPHA = 0.10` and recompute only the HP columns. What changes?
2. Replace the measurement-scale condition with distance only (`HPdist`). Which explainers would pass, and is that a better or a worse test for these data? Write two sentences.
3. Pick one applicant with a radius of 0 for KernelSHAP and read its claims. Is it a poor explanation or local structure?